# Day 07 — Embeddings, Vector DB Search & LCEL (hands-on)

Companion notebook to [`../notes.md`](../notes.md). Embeds real sentences, measures similarity with
actual numbers, builds a small local vector index, runs a real similarity search, and composes an LCEL
pipeline.

Sections 1-3 need **no API key** (embeddings run locally). Section 4's final LLM call is optional and
clearly marked.

In [1]:
import os
import numpy as np

from langchain_huggingface import HuggingFaceEmbeddings

# First run downloads this small model (~90MB) and caches it locally.
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\ramu\OneDrive\Desktop\Agentic-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. What does an embedding vector actually look like?

Day 02 explained embeddings in theory. Here are real numbers.

In [2]:
words = ["dog", "puppy", "airplane"]
vectors = embedder.embed_documents(words)

for w, v in zip(words, vectors):
    print(f"{w!r}: {len(v)} dimensions, first 5 values = {[round(x, 3) for x in v[:5]]}")

'dog': 384 dimensions, first 5 values = [-0.053, 0.014, 0.007, 0.069, -0.078]
'puppy': 384 dimensions, first 5 values = [-0.08, 0.035, 0.0, 0.031, -0.086]
'airplane': 384 dimensions, first 5 values = [0.024, 0.046, -0.072, 0.071, -0.065]


## 2. Cosine similarity — measuring how close two vectors are

Day 02: similarity near 1 = very similar meaning, near 0 = unrelated.

In [3]:
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("cosine(dog, puppy)    =", round(cosine_sim(vectors[0], vectors[1]), 4))
print("cosine(dog, airplane) =", round(cosine_sim(vectors[0], vectors[2]), 4))
print()
print("'dog' and 'puppy' land much closer together than 'dog' and 'airplane' —")
print("even though 'dog' and 'puppy' don't share a single letter.")

cosine(dog, puppy)    = 0.804
cosine(dog, airplane) = 0.3419

'dog' and 'puppy' land much closer together than 'dog' and 'airplane' —
even though 'dog' and 'puppy' don't share a single letter.


## 3. A real vector database search (FAISS, fully local)

Index a handful of support documents, then search with a question that shares **zero words** with the
best-matching document — this only works because of semantic (meaning-based) search, not keyword
matching.

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

support_docs = [
    Document(page_content="Our refund policy allows returns within 30 days of purchase."),
    Document(page_content="To reset your password, click 'forgot password' on the login page."),
    Document(page_content="Our support team is available 24/7 via chat and email."),
    Document(page_content="Shipping usually takes 3-5 business days within the country."),
]

vectorstore = FAISS.from_documents(support_docs, embedder)

query = "how do I get my money back?"
results = vectorstore.similarity_search_with_score(query, k=2)

print(f"Query: {query!r}\n")
for doc, score in results:
    print(f"  score={score:.4f}  ->  {doc.page_content}")

Query: 'how do I get my money back?'

  score=1.1219  ->  Our refund policy allows returns within 30 days of purchase.
  score=1.4922  ->  To reset your password, click 'forgot password' on the login page.


C:\Users\ramu\AppData\Local\Temp\ipykernel_20824\1763749755.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Notice the query shares no words with "refund policy" or "30 days," yet it's still the closest
match — the embeddings captured that "get my money back" and "refund" mean the same thing.

*(Lower FAISS score here means closer/more similar — FAISS's default metric is a distance, not a
similarity score, so smaller is better.)*

## 4. Composing a retrieval pipeline with LCEL

Turn the vector store into a retriever, and pipe it together with a prompt — this is the
`{"context": retriever, "question": ...} | prompt | model | parser` pattern from `notes.md` Section 2.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

### What does `format_docs` do?

A retriever gives back a **list of `Document` objects** (each has `page_content` = the text, plus
`metadata`). But the prompt needs plain **text** to put into `{context}`.

`format_docs` is the small translator between the two: it takes each document's text and joins them into
one string, with a blank line between them.

```
[Document(refund policy...), Document(password reset...)]   ──format_docs──►   "refund policy...\n\npassword reset..."
        (a list of objects)                                                         (one plain string)
```

Let's see the real before and after:

In [6]:
found = retriever.invoke("how do I get my money back?")

print("What the retriever returns:", type(found).__name__, "of", len(found), "Document objects")
print()
print("Without format_docs, the prompt would get this messy list:")
print(str(found))
print()
print("With format_docs, it gets clean text:")
print(format_docs(found))

What the retriever returns:

 list of 2 Document objects

Without format_docs, the prompt would get this messy list:
[Document(id='6ab3a632-c236-497d-ac25-a8ceb0573a4e', metadata={}, page_content='Our refund policy allows returns within 30 days of purchase.'), Document(id='ef2eec57-a439-4ed3-92ea-cf4550249cf8', metadata={}, page_content="To reset your password, click 'forgot password' on the login page.")]

With format_docs, it gets clean text:
Our refund policy allows returns within 30 days of purchase.

To reset your password, click 'forgot password' on the login page.


Much cleaner — no ids, no `Document(...)` wrappers, just the text the model should read. Now the
prompt and the dict that ties everything together:

In [7]:
prompt = ChatPromptTemplate.from_template(
    "Answer the question using only this context:\n\n{context}\n\nQuestion: {question}"
)

rag_chain_inputs = {"context": retriever | format_docs, "question": RunnablePassthrough()}

### What is `RunnablePassthrough` doing?

It is the simplest runnable there is: **it returns whatever it receives** — the same as the function
`lambda x: x`. It doesn't *find* the question; it is simply **handed** the input, like every other branch
of the dict.

Here is a tiny dict where three branches all receive the same input, `"hello"`:

In [8]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

demo = RunnableParallel({
    "shout":  RunnableLambda(lambda text: text.upper()),   # a branch that changes the input
    "length": RunnableLambda(lambda text: len(text)),      # a branch that measures the input
    "same":   RunnablePassthrough(),                       # a branch that returns the input unchanged
})

print(demo.invoke("hello"))

{'shout': 'HELLO', 'length': 5, 'same': 'hello'}


Every branch received `"hello"`. `RunnablePassthrough` just gave it back untouched — which is exactly
how the `question` key in our RAG dict ends up holding your original question.

### How does the question reach *both* `context` and `question`?

You only pass **one string** to `.invoke(...)`. But look at `rag_chain_inputs` — it's a dict with two
keys. LangChain treats a dict like this as "run every value, using the **same input**, and collect the
results under the same keys" (it quietly wraps the dict in a `RunnableParallel`).

```
"how do I get my money back?"  ──►  "context":  retriever | format_docs  →  the matching documents, as text
                               ──►  "question": RunnablePassthrough()    →  the same string, unchanged
```

A plain dict has no `.invoke`, so to peek inside we wrap it ourselves — the same thing LangChain does for
you when you use `|`:

In [9]:
from langchain_core.runnables import RunnableParallel

question = "how do I get my money back?"

step_1 = RunnableParallel(rag_chain_inputs).invoke(question)   # run only the dict part

print("keys     :", list(step_1))
print("question :", step_1["question"])
print("context  :", step_1["context"])

keys     : ['context', 'question']
question : how do I get my money back?
context  : Our refund policy allows returns within 30 days of purchase.

To reset your password, click 'forgot password' on the login page.


The one string became **two** values: `context` (found by the retriever) and `question` (passed
straight through). Now the prompt template fills its two blanks, `{context}` and `{question}`, by matching
these dict **keys** to the placeholder **names**:

In [10]:
rendered_prompt = prompt.invoke(step_1)
print(rendered_prompt.to_string())

Human: Answer the question using only this context:

Our refund policy allows returns within 30 days of purchase.

To reset your password, click 'forgot password' on the login page.

Question: how do I get my money back?


`rag_chain_inputs | prompt` does both steps in one go — that's exactly what the full chain in Section 5
uses.

## 5. (Optional) Running the full chain through a real model

This cell only runs an actual LLM call if it finds an API key in your environment. Add one to a `.env`
file in the project root (e.g. `OPENAI_API_KEY=sk-...`) to try it — otherwise this cell just explains
what it would do.

In [11]:
from dotenv import load_dotenv

load_dotenv()  # looks for a .env file up the directory tree

has_key = bool(os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY") or os.getenv("GOOGLE_API_KEY"))

if not has_key:
    print("No API key found (checked OPENAI_API_KEY / ANTHROPIC_API_KEY / GOOGLE_API_KEY).")
    print("Add one to a .env file to run the real chain:")
    print()
    print('  rag_chain = rag_chain_inputs | prompt | model | StrOutputParser()')
    print('  rag_chain.invoke("how do I get my money back?")')
else:
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    elif os.getenv("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        model = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0)
    else:
        from langchain_google_genai import ChatGoogleGenerativeAI
        model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

    rag_chain = rag_chain_inputs | prompt | model | StrOutputParser()
    answer = rag_chain.invoke("how do I get my money back?")
    print(answer)

No API key found (checked OPENAI_API_KEY / ANTHROPIC_API_KEY / GOOGLE_API_KEY).
Add one to a .env file to run the real chain:

  rag_chain = rag_chain_inputs | prompt | model | StrOutputParser()
  rag_chain.invoke("how do I get my money back?")


## Try it yourself

- Add a few more `support_docs` of your own and see how the search results change.
- Try `k=1` vs `k=3` in `as_retriever(search_kwargs={...})`.
- Swap `FAISS` for `Chroma` (`from langchain_chroma import Chroma`) — same `.from_documents(...)` API.